In [0]:
%sql
CREATE TABLE IF NOT EXISTS de_learning_workspace.silver.customers
(
    customer_id INT,

    first_name STRING,
    last_name STRING,
    email STRING,
    phone STRING,

    city STRING,
    state STRING,
    country STRING,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    _load_timestamp TIMESTAMP,
    _source_system STRING,

    _silver_processed_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
silver_source_df = spark.table(
    "de_learning_workspace.bronze.customers"
)

display(silver_source_df)

In [0]:
from pyspark.sql.functions import (
    col,
    lower,
    trim,
    coalesce,
    lit,
    current_timestamp
)


silver_customers_df = (
    silver_source_df

    # remove duplicate customer records
    .dropDuplicates(["customer_id"])

    # standardize email
    .withColumn(
        "email",
        lower(trim(col("email")))
    )

    # handle missing location
    .withColumn(
        "city",
        coalesce(col("city"), lit("Unknown"))
    )

    .withColumn(
        "state",
        coalesce(col("state"), lit("Unknown"))
    )

    .withColumn(
        "country",
        coalesce(col("country"), lit("Unknown"))
    )

    # silver processing metadata
    .withColumn(
        "_silver_processed_timestamp",
        current_timestamp()
    )
)

In [0]:
(
    silver_customers_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "de_learning_workspace.silver.customers"
    )
)


In [0]:
display(silver_customers_df)